In [1]:
import os, unicodedata
from datasets import load_dataset

c:\Users\nikhi\anaconda3\envs\slm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    - pulling the english and telugu data 

In [13]:
ROOT = "D:/Tasks/Project_SLM"
TARGET_MB = {"en": 20000}          # run english first; add en/de only if needed
#TARGET_MB = {"de": 20000}          # run german first; add en/de only if needed
SOURCES = {
    "en": [("HuggingFaceFW/fineweb-edu", "sample-10BT", "train")],
}
# SOURCES = {
#     "de": [("HuggingFaceFW/fineweb-2", "deu_Latn", "train")],
    
# }

In [14]:
def norm(t):
    t = unicodedata.normalize("NFC", (t or "")).strip().replace("\n", " ")
    return t if len(t) > 200 else None


def open_stream(repo, cfg, split):
    kw = {"data_dir": cfg} if repo.startswith("ai4bharat") else {"name": cfg}
    return load_dataset(repo, split=split, streaming=True, **kw)


def pull(lang):
    cap = TARGET_MB[lang] * 1024**2
    path = f"{ROOT}/raw/{lang}2.txt"          # NEW file — never overwrites the old one
    os.makedirs(f"{ROOT}/raw", exist_ok=True)
    written = 0
    with open(path, "w", encoding="utf-8") as f:
        for repo, cfg, split in SOURCES[lang]:
            if written >= cap:
                break
            start = written
            try:
                ds = open_stream(repo, cfg, split)
            except Exception as e:
                print(f"  SKIP {repo}/{cfg}: {type(e).__name__}: {e}")
                continue
            for row in ds:
                doc = norm(row.get("text") or row.get("content"))
                if not doc:
                    continue
                f.write(doc + "\n")
                written += len(doc.encode("utf-8"))
                if written >= cap:
                    break
                if (written - start) % (1024**3) < 4000:
                    print(f"    {written/1024**2:,.0f} MB", flush=True)
            print(f"  {repo}/{cfg}: +{(written-start)/1024**2:,.0f} MB "
                  f"(total {written/1024**2:,.0f} MB)", flush=True)
    print(f"{lang} done: {written/1024**2:,.0f} MB -> {path}")
    return written

def pull(lang):
    cap = TARGET_MB[lang] * 1024**2
    path = f"{ROOT}/raw/{lang}2.txt"
    os.makedirs(f"{ROOT}/raw", exist_ok=True)
    written, bad = 0, 0
    with open(path, "w", encoding="utf-8", errors="replace", newline="\n") as f:
        for repo, cfg, split in SOURCES[lang]:
            if written >= cap:
                break
            start = written
            try:
                ds = open_stream(repo, cfg, split)
            except Exception as e:
                print(f"  SKIP {repo}/{cfg}: {type(e).__name__}: {e}")
                continue
            for row in ds:
                doc = norm(row.get("text") or row.get("content"))
                if not doc:
                    continue
                try:
                    f.write(doc + "\n")
                except (OSError, UnicodeError) as e:
                    bad += 1
                    if bad < 5:
                        print(f"    write failed ({type(e).__name__}), skipping row")
                    if bad > 1000:
                        raise                      # drive is gone, not a data issue
                    continue
                written += len(doc.encode("utf-8"))
                if written >= cap:
                    break
                if (written - start) % (1024**3) < 4000:
                    print(f"    {written/1024**2:,.0f} MB", flush=True)
            print(f"  {repo}/{cfg}: +{(written-start)/1024**2:,.0f} MB "
                  f"(total {written/1024**2:,.0f} MB)", flush=True)
    print(f"{lang} done: {written/1024**2:,.0f} MB, {bad} rows skipped -> {path}")
    return written

In [15]:
if __name__ == "__main__":
    for lang in TARGET_MB:
        print(f"\n=== {lang} ===")
        pull(lang)


=== en ===
    0 MB
    2,048 MB
    3,072 MB
    3,072 MB
    4,096 MB
    5,120 MB
    5,120 MB
    6,144 MB
    7,168 MB
    8,192 MB
    10,240 MB
    12,288 MB
    12,288 MB
    13,312 MB
    14,336 MB
    16,384 MB
    16,384 MB
    17,408 MB
    17,408 MB
    18,432 MB
    18,432 MB
    18,432 MB
  HuggingFaceFW/fineweb-edu/sample-10BT: +20,000 MB (total 20,000 MB)
en done: 20,000 MB, 0 rows skipped -> D:/Tasks/Project_SLM/raw/en2.txt


    - pulling only telugu data 

In [4]:


ROOT = "C:/Project_Slm"
TARGET_MB = {"te": 27000}          # run Telugu first; add en/de only if needed

SOURCES = {
    "en": [("HuggingFaceFW/fineweb-edu", "sample-10BT", "train")],
    "de": [("HuggingFaceFW/fineweb-2", "deu_Latn", "train")],
    "te": [("HuggingFaceFW/fineweb-2", "tel_Telu", "train"),
           ("ai4bharat/sangraha", "verified/tel", "train"),
           ("uonlp/CulturaX", "te", "train"),
           ("ai4bharat/IndicCorpV2", "te", "train")],
}


def norm(t):
    t = unicodedata.normalize("NFC", (t or "")).strip().replace("\n", " ")
    return t if len(t) > 200 else None


def open_stream(repo, cfg, split):
    kw = {"data_dir": cfg} if repo.startswith("ai4bharat") else {"name": cfg}
    return load_dataset(repo, split=split, streaming=True, **kw)


def pull(lang):
    cap = TARGET_MB[lang] * 1024**2
    path = f"{ROOT}/raw/{lang}2.txt"          # NEW file — never overwrites the old one
    os.makedirs(f"{ROOT}/raw", exist_ok=True)
    written = 0
    with open(path, "w", encoding="utf-8") as f:
        for repo, cfg, split in SOURCES[lang]:
            if written >= cap:
                break
            start = written
            try:
                ds = open_stream(repo, cfg, split)
            except Exception as e:
                print(f"  SKIP {repo}/{cfg}: {type(e).__name__}: {e}")
                continue
            for row in ds:
                doc = norm(row.get("text") or row.get("content"))
                if not doc:
                    continue
                f.write(doc + "\n")
                written += len(doc.encode("utf-8"))
                if written >= cap:
                    break
                if (written - start) % (1024**3) < 4000:
                    print(f"    {written/1024**2:,.0f} MB", flush=True)
            print(f"  {repo}/{cfg}: +{(written-start)/1024**2:,.0f} MB "
                  f"(total {written/1024**2:,.0f} MB)", flush=True)
    print(f"{lang} done: {written/1024**2:,.0f} MB -> {path}")
    return written

def pull(lang):
    cap = TARGET_MB[lang] * 1024**2
    path = f"{ROOT}/raw/{lang}2.txt"
    os.makedirs(f"{ROOT}/raw", exist_ok=True)
    written, bad = 0, 0
    with open(path, "w", encoding="utf-8", errors="replace", newline="\n") as f:
        for repo, cfg, split in SOURCES[lang]:
            if written >= cap:
                break
            start = written
            try:
                ds = open_stream(repo, cfg, split)
            except Exception as e:
                print(f"  SKIP {repo}/{cfg}: {type(e).__name__}: {e}")
                continue
            for row in ds:
                doc = norm(row.get("text") or row.get("content"))
                if not doc:
                    continue
                try:
                    f.write(doc + "\n")
                except (OSError, UnicodeError) as e:
                    bad += 1
                    if bad < 5:
                        print(f"    write failed ({type(e).__name__}), skipping row")
                    if bad > 1000:
                        raise                      # drive is gone, not a data issue
                    continue
                written += len(doc.encode("utf-8"))
                if written >= cap:
                    break
                if (written - start) % (1024**3) < 4000:
                    print(f"    {written/1024**2:,.0f} MB", flush=True)
            print(f"  {repo}/{cfg}: +{(written-start)/1024**2:,.0f} MB "
                  f"(total {written/1024**2:,.0f} MB)", flush=True)
    print(f"{lang} done: {written/1024**2:,.0f} MB, {bad} rows skipped -> {path}")
    return written




In [5]:
if __name__ == "__main__":
    for lang in TARGET_MB:
        print(f"\n=== {lang} ===")
        pull(lang)


=== te ===
    0 MB
    1,024 MB
    2,048 MB
    5,120 MB
    9,216 MB
    11,264 MB
    12,288 MB
    14,336 MB
  HuggingFaceFW/fineweb-2/tel_Telu: +14,761 MB (total 14,761 MB)
    15,785 MB
    15,785 MB
    18,857 MB
    18,857 MB
    22,953 MB
    23,977 MB
    23,977 MB
    25,001 MB
    25,001 MB
    26,025 MB
  ai4bharat/sangraha/verified/tel: +12,239 MB (total 27,000 MB)
te done: 27,000 MB, 0 rows skipped -> C:/Project_Slm/raw/te2.txt


In [ ]:
-5*

In [ ]:
TARGET_MB = {"en": 20000, "de": 17000, "te": 27000}   # per language, on disk
OUT = "E:/Project_SLM/raw"
ROOT = "E:/Project_SLM"

CAP = 2000 * 1024**2          # 2 GB
os.makedirs(f"{ROOT}/raw", exist_ok=True)
SOURCES = {
    "en": [("HuggingFaceFW/fineweb-edu", "sample-10BT", "train")],
    "de": [("HuggingFaceFW/fineweb-2", "deu_Latn", "train")],
    "te": [("HuggingFaceFW/fineweb-2", "tel_Telu", "train"),
           ("ai4bharat/sangraha", "verified/tel", "train"),
           ("uonlp/CulturaX", "te", "train"),
           ("ai4bharat/IndicCorpV2", "te", "train")],
}

In [ ]:

def clean(t):
    t = unicodedata.normalize("NFC", t).strip()
    return t.replace("\n", " ") if len(t) > 200 else None

os.makedirs(OUT, exist_ok=True)
for lang, srcs in SOURCES.items():
    path, cap, written = f"{OUT}/{lang}2.txt", TARGET_MB[lang] * 1024**2, 0
    with open(path, "w", encoding="utf-8") as f:
        for repo, cfg, split in srcs:
            if written >= cap:
                break
            kw = {"data_dir": cfg} if repo.startswith("ai4bharat") else {"name": cfg}
            ds = load_dataset(repo, split=split, streaming=True, **kw)
            for row in ds:
                doc = clean(row.get("text") or row.get("content") or "")
                if not doc:
                    continue
                f.write(doc + "\n")
                written += len(doc.encode("utf-8"))
                if written >= cap:
                    break
            print(f"{lang} <- {repo}/{cfg}: {written/1024**2:.0f} MB")
    print(f"{lang} done: {written/1024**2:.0f} MB -> {path}")

ds = load_dataset("ai4bharat/sangraha", data_dir="verified/tel",
                  split="train", streaming=True)

written = 0
with open(f"{ROOT}/raw/te_sangraha.txt", "w", encoding="utf-8") as f:
    for row in ds:
        doc = row.get("text") or row.get("content") or ""
        doc = unicodedata.normalize("NFC", doc).strip().replace("\n", " ")
        if len(doc) < 200:
            continue
        f.write(doc + "\n")
        written += len(doc.encode("utf-8"))
        if written >= CAP:
            break
print(f"done: {written/1e9:.2f} GB -> {ROOT}/raw/te_sangraha.txt")

In [ ]:
(1 << 61) - 1 